<a href="https://colab.research.google.com/github/cauedasneves/cienciadedados3tds/blob/main/C%C3%B3pia_de_3ds_Analise_exploratoria_dados_faturamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 **ANÁLISE EXPLORATÓRIA DE DADOS DE FATURAMENTO**  

⚠️ATENÇÃO!!!⚠️ FAÇA UMA CÓPIA DESTE ARQUIVO NO SEU GOOGLE DRIVE ANTES DE COMEÇAR.  


 Clique no ícone de **Pasta** na lateral esquerda do Colab  
 e faça upload do arquivo **12-03 - Dashboard-base.xlsx**.

**Célula 1: Importação das Bibliotecas**

Nesta etapa, carregamos o pandas para tratar os dados e o plotly para criar gráficos interativos e modernos.

In [1]:
import pandas as pd
import plotly.express as px

**Célula 2: Carregamento da Base de Dados e Amostra**  

*Nota: Certifique-se de que o arquivo "12-03 - Dashboard-base.xlsx" foi carregado na pasta lateral do Colab.*




In [3]:
df = pd.read_excel('12-03 - Dashboard-base.xlsx')
df.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Data,Ano,Mês,Vendedor,Cliente,Região,Produto,Valor
0,2019-01-01,2019,jan,Paulo,Thamires Bastos,Sudeste,Python,450
1,2019-01-01,2019,jan,Paulo,Jessika Mineiro,Centro-Oeste,Excel,500
2,2019-01-02,2019,jan,Diego,Glenda Jalles,Norte,VBA,600
3,2019-01-02,2019,jan,Diego,Eugênio Mattos,Sul,Python,450
4,2019-01-02,2019,jan,Alon,Yasmine Gomes,Norte,VBA,600


**Célula 3: Estrutura da Base de Dados**

Aqui verificamos os tipos de colunas e se há necessidade de ajustes (como converter a Data).

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3669 entries, 0 to 3668
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Data      3669 non-null   datetime64[ns]
 1   Ano       3669 non-null   int64         
 2   Mês       3669 non-null   object        
 3   Vendedor  3669 non-null   object        
 4   Cliente   3669 non-null   object        
 5   Região    3669 non-null   object        
 6   Produto   3669 non-null   object        
 7   Valor     3669 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 229.4+ KB


**Célula 4: Resumo Estatístico**

Uma visão geral sobre os valores financeiros da planilha.

In [5]:
df.describe()

,Data,Ano,Valor
count,3669,3669.000000,3669.000000
mean,2020-07-01 13:24:11.185609216,2019.994821,462.264922
min,2019-01-01 00:00:00,2019.000000,300.000000
25%,2019-10-04 00:00:00,2019.000000,300.000000
50%,2020-06-27 00:00:00,2020.000000,450.000000
75%,2021-04-01 00:00:00,2021.000000,500.000000
max,2021-12-31 00:00:00,2021.000000,600.000000
std,NaN,0.813749,108.207461


**Célula 5: Gráfico 1 - Faturamento Total por Vendedor**

Este gráfico de barras horizontais ajuda a identificar rapidamente quem são os melhores vendedores, similar ao dashboard original.

In [6]:
faturamento_por_vendedor = df.groupby('Vendedor')['Valor'].sum().reset_index()
faturamento_por_vendedor = faturamento_por_vendedor.sort_values(by='Valor', ascending=True)

fig = px.bar(
    faturamento_por_vendedor,
    x='Valor',
    y='Vendedor',
    orientation='h',
    title='Faturamento Total por Vendedor',
    labels={'Valor': 'Faturamento Total (R$)', 'Vendedor': 'Vendedor'},
    text='Valor',
    color='Valor',
    color_continuous_scale=px.colors.sequential.Plotly3
)

fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
fig.show()

**Célula 6: Gráfico 2 - Distribuição por Região**

Um gráfico de rosca para entender a participação de cada mercado.

In [7]:
faturamento_por_regiao = df.groupby('Região')['Valor'].sum().reset_index()

fig = px.pie(
    faturamento_por_regiao,
    values='Valor',
    names='Região',
    title='Faturamento Total por Região',
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.Plotly3
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

**Célula 7: Gráfico 3 - Evolução Mensal do Faturamento**

Gráfico de linhas para visualizar a sazonalidade e o crescimento ao longo dos meses.

In [8]:
faturamento_mensal = df.groupby(['Ano', 'Mês'])['Valor'].sum().reset_index()

# Criar uma coluna de ordem para os meses para garantir a ordenação correta no gráfico
mes_map = {'jan': 1, 'fev': 2, 'mar': 3, 'abr': 4, 'mai': 5, 'jun': 6, 'jul': 7, 'ago': 8, 'set': 9, 'out': 10, 'nov': 11, 'dez': 12}
faturamento_mensal['Mes_Num'] = faturamento_mensal['Mês'].map(mes_map)
faturamento_mensal = faturamento_mensal.sort_values(by=['Ano', 'Mes_Num'])

# Criar uma coluna de 'Ano-Mês' para o eixo x do gráfico
faturamento_mensal['Ano_Mes'] = faturamento_mensal['Ano'].astype(str) + '-' + faturamento_mensal['Mês']

fig = px.line(
    faturamento_mensal,
    x='Ano_Mes',
    y='Valor',
    title='Evolução Mensal do Faturamento',
    labels={'Valor': 'Faturamento Total (R$)', 'Ano_Mes': 'Ano - Mês'},
    markers=True, # Adiciona marcadores nos pontos dos meses
    color_discrete_sequence=px.colors.sequential.Plotly3
)

fig.update_xaxes(tickangle=45)
fig.show()

**Célula 8: Gráfico 4 - Faturamento por Produto e Região**

Gráfico de histogramas para visualizar e comparar o faturamento por produto em cada região do país.

In [9]:
faturamento_por_produto_regiao = df.groupby(['Região', 'Produto'])['Valor'].sum().reset_index()

fig = px.histogram(
    faturamento_por_produto_regiao,
    x='Produto',
    y='Valor',
    color='Região',
    barmode='group', # Para comparar produtos dentro de cada região
    title='Faturamento por Produto e Região',
    labels={'Valor': 'Faturamento Total (R$)', 'Produto': 'Produto', 'Região': 'Região'},
    color_discrete_sequence=px.colors.sequential.Plotly3
)

fig.update_layout(xaxis_title='Produto', yaxis_title='Faturamento Total (R$)')
fig.show()

***😃A análise exploratória de dados está concluída!***